<h1 align='center'> How much of the beginning should I cut?

Import necessary modules:

In [ ]:
import copy
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import pandas as pd
import numpy as np
from glob import glob
import plumed
from scipy.stats import zscore

%matplotlib inline

# Load COLVAR file

In [ ]:
# Load the COLVAR files
path_to_colvar_file = "THE PATH TO YOUR COLVAR FILE"
COLVAR = plumed.read_as_pandas(path_to_colvar_file)

# See when to cut beginning

In [ ]:
def compute_and_plot_sliding_window_average(df, time_window, df_whole, axis, window_der=50):
    # Ensure DataFrame is sorted by time
    df_whole = df_whole.sort_values(by='time').reset_index(drop=True)
    df = df.sort_values(by='time').reset_index(drop=True)
    
    # Calculate the sliding window size in terms of the number of points
    time_step = df_whole['time'].iloc[1] - df_whole['time'].iloc[0]  # Assumes uniform time step
    
    window_size = int(time_window * 1000)

    
    # Calculate the sliding window average
    df['opes.bias_smooth'] = df['opes.bias'].rolling(window=window_size, min_periods=1).mean()
    
    # Calculate the first derivative of the smoothed data
    df['derivative'] = np.gradient(df['opes.bias_smooth'], df['time'])
    
    # Calculate the absolute values of the derivatives
    df['abs_derivative'] = np.abs(df['derivative'])
    
    # Plot derivative distribution
    #df[['derivative']].hist(bins=100)
    
    # Identify outliers in the distribution of absolute derivatives using Z-scores
    z_scores = zscore(df['derivative'])
    outliers = np.abs(z_scores) > 3  # Common threshold for outliers is 3 standard deviations
    outlier_indices = np.where(outliers)[0]
    
    min_outlier = df['abs_derivative'].iloc[outlier_indices].values.min()
    
    # Calculate the number of windows
    n_windows = len(np.unique(df_whole['time'])) // (window_der * 1000)
    
    # Create initial window boundaries based on the desired number of windows
    total_points = len(df_whole)
    points_per_window = total_points // n_windows
    window_starts = [i * points_per_window for i in range(n_windows)]
    
    window_indices = np.digitize(df['time'], bins=[df_whole['time'].iloc[i] for i in window_starts])
    df['window_indices'] = window_indices
    
    
    plateau_start_time = None
    for i in range(1, len(window_starts)):
        window_df = df[df['window_indices'] == i]
        if np.sum(window_df['abs_derivative']>=min_outlier) == 0:
            plateau_start_time = window_starts[i]
            break

    
    
    # Plotting
    axis.scatter(df['time']//1000, df['opes.bias'], label='Data points', s=10, color='gray', alpha=0.5)
    axis.plot(df['time']//1000, df['opes.bias_smooth'], label='Smoothed Data', color='red', linewidth=2)
    
    if plateau_start_time is not None:
        axis.axvline(plateau_start_time//1000, color='blue', linestyle='--', label='Plateau Start')
    
    axis.set_xlabel('Time (ns)', fontsize = label_size*0.7)
    axis.set_ylabel('OPES Bias', fontsize = label_size*0.7)
    axis.legend(bbox_to_anchor=(1, 1), loc='upper left', 
               ncol=1, fontsize = label_size*0.7, framealpha=0)

    
    axis.text(400, -30, f'The plateau starts at time: {plateau_start_time/1000} ns', fontsize = label_size*0.7)


In [ ]:
def clean_colvar(df):
    """
    Takes a DataFrame and keeps only the last row for each repeated value
    in the 'time' column.
    
    Parameters:
    df (pd.DataFrame): The input DataFrame.
    
    Returns:
    pd.DataFrame: A DataFrame with only the last row for each repeated 'time' value.
    """
    # Drop duplicates, keeping only the last occurrence
    result_df = df.drop_duplicates(subset='time', keep='last')
    return result_df

The bound state is defined as a surrounding of the starting configuration, but you can change it
the unbound state is defined by hand
time window and window der can be changed according to the situation

In [ ]:
time_window= 0.5 #ns
window_der = 100 #100#ns


In [ ]:
fig, axs= plt.subplots(nrows=2, ncols=1, sharey=False, sharex=True, figsize=(8,16))
axs = axs.flatten()

image_filename = "where_to_cut_beginning.png"

df_tmp = clean_colvar(copy.deepcopy(COLVAR))

# bound

axs[0].set_title("Bound")

compute_and_plot_sliding_window_average(df_tmp[(df_tmp['pp.proj'] > df_tmp['pp.proj'].values[0] - 0.1) & (df_tmp['pp.proj'] < df_tmp['pp.proj'].values[0] + 0.1) & (df_tmp['cmap'] > df_tmp['cmap'].values[0] - 0.05) & (df_tmp['cmap'] < df_tmp['cmap'].values[0] + 0.05)],
                                            time_window, df_tmp, axis=axs[0], window_der=window_der)
axs[0].set_ylim(bottom=-31)

df_tmp = None


# Unbound

df_tmp = clean_colvar(copy.deepcopy(COLVAR))

axs[1].set_title("Unbound")

unbound_proj_lb = 3.1 # the lower value of the projection for the unbound state
unbound_proj_ub = 3.2 # higher value of the projection for the unbound state

compute_and_plot_sliding_window_average(df_tmp[(df_tmp['pp.proj'] > unbound_proj_lb) & (df_tmp['pp.proj'] < unbound_proj_ub) & (df_tmp['cmap'] < 0.1)  ],
                                            time_window, df_tmp, axis=axs[1], window_der=window_der)

axs[1].set_ylim(bottom=-31)

df_tmp = None


plt.savefig(image_filename, bbox_inches='tight', transparent=True, dpi = 500)

# Make vectorial image
plt.savefig(str(image_filename).split(".")[0] + ".svg",
            bbox_inches='tight', transparent=True, format = 'svg', dpi=500)